# RBF with Ellipsoid Constraint
## Reproducible Workflow — Li & Griffiths (CGF 2004)

This notebook demonstrates the Python implementation of the RBF with Ellipsoid Constraint
algorithm described in:

> Li, Q. and Griffiths, J. G. (2004).  
> *Radial basis functions for surface reconstruction from unorganised point clouds with
> applications to bone reconstruction.*  
> Computer Graphics Forum, 23(1), 67–78. Wiley-Blackwell.  
> DOI: [10.1111/j.1467-8659.2004.00005.x](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1467-8659.2004.00005.x)

### Contents
1. [Install & import](#1-install--import)
2. [Synthetic data generation](#2-synthetic-data-generation)
3. [Fitting an axis-aligned ellipsoid](#3-fitting-an-axis-aligned-ellipsoid)
4. [Fitting a rotated ellipsoid](#4-fitting-a-rotated-ellipsoid)
5. [Fitting from CSV datasets](#5-fitting-from-csv-datasets)
6. [Residual analysis](#6-residual-analysis)
7. [Visualisation](#7-visualisation)


## 1. Install & import

In [ ]:
# If running from the repository root, the package is importable directly.
# Otherwise: pip install -e .
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend for CI / notebook export
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from rbf_ellipsoid_constraint import (
    fit_rbf_ellipsoid_linear,
    evaluate_model_linear,
    generate_ellipsoid_points,
    load_point_cloud,
)
print("rbf_ellipsoid_constraint imported successfully")

## 2. Synthetic data generation

The  function creates quasi-uniformly distributed points on an ellipsoid surface,
optionally perturbed with Gaussian noise.

In [ ]:
TRUE_CENTRE = (1.0, 2.0, 3.0)
TRUE_RADII  = (5.0, 3.0, 2.0)

pts = generate_ellipsoid_points(
    centre=TRUE_CENTRE,
    radii=TRUE_RADII,
    n_points=300,
    noise_std=0.05,
    seed=42,
)

print(f"Shape : {pts.shape}")
print(f"x range: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"y range: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"z range: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

## 3. Fitting an axis-aligned ellipsoid

 solves the generalised eigenvalue problem with ellipsoid constraint
and returns RBF weights, polynomial coefficients, centroid, and scale.

In [ ]:
result = fit_rbf_ellipsoid_linear(pts, smooth=0.05)

if result is None:
    print("Fit failed: no valid eigenvalue found.")
else:
    alpha, beta, cent, scale = result
    norm_pts = (pts - cent) / scale
    vals = evaluate_model_linear(norm_pts, norm_pts, alpha, beta)
    print(f"Centroid : {cent.round(4)}")
    print(f"Scale    : {scale:.4f}")
    print(f"Mean |F| : {np.mean(np.abs(vals)):.6f}  (should be close to 0)")
    print(f"Max  |F| : {np.max(np.abs(vals)):.6f}")

## 4. Fitting a rotated ellipsoid

The algorithm handles arbitrary rotations naturally.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler("xyz", [30, 45, 60], degrees=True).as_matrix()
pts_rot = generate_ellipsoid_points(
    centre=(2, -1, 4), radii=(6, 4, 2),
    rotation=R, n_points=500, noise_std=0.05, seed=123,
)

result_rot = fit_rbf_ellipsoid_linear(pts_rot, smooth=0.05)
if result_rot is not None:
    alpha_r, beta_r, cent_r, scale_r = result_rot
    norm_rot = (pts_rot - cent_r) / scale_r
    vals_rot = evaluate_model_linear(norm_rot, norm_rot, alpha_r, beta_r)
    print(f"Centroid : {cent_r.round(4)}")
    print(f"Mean |F| : {np.mean(np.abs(vals_rot)):.6f}")

## 5. Fitting from CSV datasets

In [ ]:
data_dir = os.path.join("..", "data")
csv_files = [
    "synthetic_ellipsoid_low_noise.csv",
    "synthetic_ellipsoid_rotated.csv",
    "synthetic_sphere_like.csv",
]

for fname in csv_files:
    path = os.path.join(data_dir, fname)
    data = np.loadtxt(path, delimiter=",", skiprows=1)
    pts_csv = data[:, :3]
    res = fit_rbf_ellipsoid_linear(pts_csv, smooth=0.05)
    if res is not None:
        a, b, c, s = res
        n = (pts_csv - c) / s
        v = evaluate_model_linear(n, n, a, b)
        print(f"{fname}: centroid={c.round(3)}, mean|F|={np.mean(np.abs(v)):.4f}")
    else:
        print(f"{fname}: fit failed")

## 6. Residual analysis

The implicit function value  should be close to zero for points on the fitted surface.

In [ ]:
pts_test = generate_ellipsoid_points(radii=(3, 2, 1), n_points=300, noise_std=0.01, seed=0)
res_test = fit_rbf_ellipsoid_linear(pts_test, smooth=0.05)
if res_test is not None:
    a_t, b_t, c_t, s_t = res_test
    n_t = (pts_test - c_t) / s_t
    v_t = evaluate_model_linear(n_t, n_t, a_t, b_t)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(np.abs(v_t), bins=30, color="steelblue", edgecolor="white")
    ax.set_xlabel("|F(q)|")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of |F| on surface points")
    plt.tight_layout()
    plt.savefig("/tmp/residuals.png", dpi=100)
    plt.show()
    print(f"Mean |F| = {np.mean(np.abs(v_t)):.6f}")

## 7. Visualisation

Scatter plot of the fitted point cloud.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=4, alpha=0.5, c="steelblue", label="Points")
ax.set_title("RBF with Ellipsoid Constraint (Li & Griffiths, CGF 2004)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/rbf_ellipsoid_fit.png", dpi=100)
plt.show()